# 08 — Hyperparameter Tuning, Threshold Calibration, and Evaluation

## Objective

Select model configurations using the designated development data, calibrate anomaly-score thresholds without using the final test labels, and evaluate the finalized models on the held-out test data.

This notebook separates three activities:

1. **Hyperparameter tuning** — select model configurations using the designated development data.
2. **Threshold calibration** — determine anomaly-detection thresholds from normal development data.
3. **Final evaluation** — apply the selected models and fixed thresholds to the held-out evaluation data.

The final test/attack labels are not used for hyperparameter selection or threshold calibration.

## Evaluation Principle

An anomaly-detection model produces a numerical score that represents how unusual an observation or sequence is. A threshold converts this score into a binary prediction.

Let $s_i$ denote the anomaly score for observation or sequence $i$, and let $\tau$ denote the selected threshold. The predicted label is

$$
\hat{y}_i =
\begin{cases}
1, & s_i > \tau,\\
0, & s_i \leq \tau,
\end{cases}
$$

where $\hat{y}_i = 1$ indicates a detected anomaly and $\hat{y}_i = 0$ indicates no detected anomaly.

### Reconstruction Error

For an autoencoder, reconstruction error measures the difference between the input and its reconstructed output.

Let $X_i \in \mathbb{R}^{L \times F}$ be input sequence $i$, and let $\hat{X}_i \in \mathbb{R}^{L \times F}$ be its reconstruction, where:

- $L$ is the sequence length;
- $F$ is the number of input features;
- $x_{i,t,f}$ is the observed value at timestep $t$ and feature $f$;
- $\hat{x}_{i,t,f}$ is the corresponding reconstructed value.

The mean squared reconstruction error for sequence $i$ is

$$
s_i =
\frac{1}{L F}
\sum_{t=1}^{L}
\sum_{f=1}^{F}
\left( x_{i,t,f} - \hat{x}_{i,t,f} \right)^2 .
$$

A larger value indicates a greater reconstruction discrepancy. However, a large reconstruction error is not, by itself, proof of an attack.

### Important Implementation Distinction

The equation above defines a sequence-level reconstruction score. It does not establish that every model in this project uses the same scoring procedure.

Before evaluating the models, the implementation must specify whether each method produces scores per observation, per timestep, or per sequence. For sliding windows, the relationship between sequence scores and the original timestamps must also be defined to ensure that predictions and ground-truth labels are aligned correctly.

Thresholds must be calibrated using the designated normal development data and fixed before final evaluation. Final test labels must not be used to select thresholds or tune model configurations.

## Data Separation

The project preserves the chronological structure of the SWaT time series.

Data used for:

- model training,
- hyperparameter selection,
- threshold calibration,
- and final evaluation

must remain explicitly separated.

In particular, final evaluation labels are not used to select hyperparameters or thresholds. This prevents information from the final evaluation set from influencing model decisions.

## Model Scope

The evaluation covers the anomaly-detection methods defined for this project:

### Baselines

- Rule-Based Detection
- Rolling Statistical Detection
- Simple Statistical Threshold

### Traditional Machine Learning

- Isolation Forest
- One-Class SVM
- Local Outlier Factor in novelty-detection mode

### Deep Learning

- LSTM Autoencoder
- TCN Autoencoder
- USAD

The preliminary TCN and USAD training results from Notebook 07 are treated as experimental evidence, not as final model-selection results.

## Evaluation Metrics

Performance will be assessed using appropriate classification and detection metrics, including:

- Precision
- Recall
- F1-score
- Confusion matrix
- False-positive and false-negative counts

Where appropriate, event-level behavior will also be examined in the subsequent error and event analysis stage.

No single metric will be interpreted in isolation.